# Next-Concept Model (NCM) on Tiny Shakespeare — Kaggle 2×T4
Re-implementation **from the paper** (arXiv 2610.14162, Stanford Hazy Research) + a size-matched nanoGPT-style AR baseline.

**What an NCM is.** Stage 1: a multiscale residual VQ-VAE turns a 128-token block into K=12 levels of discrete codes (level k has ℓ_k codes, ℓ_1=1 … ℓ_K=128), trained with nested dropout so early levels are a coarse "gist". Stage 2: a frozen tokenizer + one transformer that predicts **all ℓ_k codes of a level in parallel**, coarse→fine (K forward passes = NFE per sample), using block-diagonal attention (levels in one sequence), conditioning-corruption (random input codes) and prefix-LM prompts. Tokens come from the tokenizer decoder by argmax.

**Verified vs assumed** (I read the paper PDF; the official repo is *evaluation-only* — no training code — and GitHub source files were not readable by my tools, so **no repo API is used**):

| Verified in paper | Assumed / my choice (flagged) |
|---|---|
| Architecture & losses (Eq. 7–10), K=12, encoder = embedding only, 1-block decoder, EMA codebook, nested-dropout p=0.5, corruption mean 0.5 (linearly decreasing), loss weights α, prefix-LM prompts, argmax token decoding, GPT-2-large gen-PPL protocol | Linear level schedule (paper uses it in an ablation; default unknown), corruption shape 0.8→0.2, β=0.25, interp+Conv1d for R_k, avg-pool for Down/resize, learned slot embeddings (not RoPE), per-level output heads, dead-code restart, per-sample nested cutoff, sampling temperature 1.0, all sizes/LRs (shrunk for 1 MB data) |

Nothing here was run on a GPU by me (no torch in my sandbox): the code is syntax-checked only. Use `SMOKE=True` first (~3–5 min) to catch bugs cheaply.

In [ ]:
import os, json, sys, time, subprocess
from pathlib import Path
SMOKE = False          # True = tiny end-to-end dry run (use before the real run)
WORK = Path('/kaggle/working/ncm') if Path('/kaggle/working').exists() else Path('./ncm_run')
WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK)
FRESH = True           # True = wipe old tok/gen/ar checkpoints+logs (stage scripts auto-resume from last.pt otherwise). Set False to resume a crashed run.
if FRESH:
    import shutil
    for d in ('tok', 'gen', 'ar', 'data'): shutil.rmtree(WORK/d, ignore_errors=True)
    for f in WORK.glob('*.log'): f.unlink()

CFG = dict(work=str(WORK), device='auto', seed=42,
  # data
  block_len=128, max_prompt=32, val_frac=0.1, bpe_vocab=2048, val_stride=128,
  util_val_stride=16, util_train_windows=8192, eval_train_windows=256,
  # stage 1 tokenizer (~5M params)
  K=12, level_schedule='linear', codebook_size=1024, d_lat=192, dec_layers=1, dec_heads=3,
  tok_dropout=0.3, ema_decay=0.95, beta_commit=0.25, nested_dropout=0.5,
  tok_batch=256, tok_lr=5e-4, tok_lr_floor=5e-5, tok_warmup=300, tok_wd=0.01, tok_betas=[0.9,0.95],
  tok_steps=6000, tok_max_minutes=30, tok_eval_every=500,
  # stage 2 generator (~15M params)
  gen_layers=6, gen_d=384, gen_heads=6, gen_dropout=0.2, gen_batch=128, gen_lr=3e-4, gen_lr_floor=3e-5,
  gen_warmup=300, gen_wd=0.1, gen_betas=[0.9,0.95], gen_steps=4500, gen_max_minutes=115,
  eval_every=250, patience=8, cond_prob=0.5, corrupt_hi=0.8, corrupt_lo=0.2, alpha=0.0,
  # AR baseline (layers auto-matched to generator params)
  ar_layers=None, ar_dropout=0.2, ar_batch=128, ar_lr=6e-4, ar_lr_floor=6e-5, ar_warmup=200, ar_wd=0.1,
  ar_steps=4000, ar_max_minutes=25, ar_eval_every=50,
  log_every=100, ckpt_every=500,
  # evaluation
  ref_models=['gpt2-large','gpt2'], ref_path=None,   # ref_path: offline Kaggle dataset dir with a HF causal LM
  n_samples=1024, n_show=20, temps=[1.0, 0.8], bench_bs=[16,128], bench_repeats=2)

if SMOKE:
    CFG.update(block_len=32, max_prompt=8, bpe_vocab=256, K=4, codebook_size=64, d_lat=32, dec_heads=2,
      gen_d=64, gen_layers=2, gen_heads=2, tok_batch=16, gen_batch=16, ar_batch=16, tok_steps=40, gen_steps=40, ar_steps=40,
      tok_eval_every=20, eval_every=20, ar_eval_every=20, ckpt_every=20, log_every=10, tok_warmup=5, gen_warmup=5, ar_warmup=5,
      val_stride=32, util_val_stride=8, util_train_windows=64, eval_train_windows=32, patience=3,
      tok_max_minutes=5, gen_max_minutes=5, ar_max_minutes=5, n_samples=32, n_show=4, temps=[1.0], bench_bs=[8],
      bench_repeats=1, ref_models=['gpt2'])
def save_cfg(): json.dump(CFG, open(WORK/'config.json','w'), indent=1)
T_START = time.time(); print('work dir:', WORK, '| SMOKE =', SMOKE)

In [ ]:
%%writefile ncm_lib.py
"""
ncm_lib.py -- compact re-implementation of Next-Concept Modeling (arXiv:2610.14162) and a
nanoGPT-style baseline, for Tiny Shakespeare. Written FROM THE PAPER: the official repo
(HazyResearch/ncm) is evaluation-only (no training code), so nothing here calls its API.
See the notebook header for which details are paper-verified vs. assumed.
"""
import os, sys, json, math, time, argparse, contextlib, random
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F


# ----------------------------------------------------------------------------- utils
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def pick_device(cfg):
    if cfg.get("device", "auto") == "cpu" or not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device("cuda")


def amp_ctx(dev, enabled=True):
    """fp16 autocast on CUDA (T4 has no fast bf16); no-op on CPU."""
    if dev.type == "cuda" and enabled:
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


def make_scaler(dev):
    en = dev.type == "cuda"
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=en)


def lr_at(step, total, warmup, lr, floor):
    if step < warmup:
        return lr * (step + 1) / max(1, warmup)
    p = min(1.0, (step - warmup) / max(1, total - warmup))
    return floor + 0.5 * (lr - floor) * (1 + math.cos(math.pi * p))


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def level_lengths(K, L, schedule):
    """l_1 < ... < l_K = L. Paper (App. F.1.2): K levels from 1 code to L codes, rounded;
    schedules compared there: linear/quadratic/sqrt/log/exp/cosine. (Exact formulas assumed.)"""
    t = np.linspace(0, 1, K) if K > 1 else np.array([1.0])
    f = {"linear": t, "quadratic": t ** 2, "sqrt": np.sqrt(t), "cosine": (1 - np.cos(np.pi * t)) / 2}[schedule]
    lens = [int(x) for x in np.rint(1 + (L - 1) * f)]
    for i in range(1, K):
        lens[i] = max(lens[i], lens[i - 1] + 1)
    lens[-1] = L
    assert all(lens[i] < lens[i + 1] for i in range(K - 1)), lens
    return lens


def save_atomic(obj, path):
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def tload(path, dev):
    try:
        return torch.load(path, map_location=dev, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=dev)


def jdump(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f)


# ----------------------------------------------------------------------------- data
class TextData:
    """Token ids live on the GPU (a few hundred K ints) -> no DataLoader, ~zero CPU RAM."""
    def __init__(self, data_dir, dev):
        d = Path(data_dir)
        self.train = torch.from_numpy(np.load(d / "train_ids.npy").astype(np.int64)).to(dev)
        self.val = torch.from_numpy(np.load(d / "val_ids.npy").astype(np.int64)).to(dev)
        self.dev = dev

    def train_windows(self, B, length, gen=None):
        n = self.train.numel() - length
        st = torch.randint(0, n, (B,), generator=gen, device="cpu" if gen is not None else self.dev).to(self.dev)
        return self.train[st[:, None] + torch.arange(length, device=self.dev)[None]]

    def val_windows(self, length, stride):
        st = torch.arange(0, self.val.numel() - length + 1, stride, device=self.dev)
        return self.val[st[:, None] + torch.arange(length, device=self.dev)[None]]


def tok_cfg(cfg):
    L = cfg["block_len"]
    return dict(vocab=cfg["vocab"], L=L, K=cfg["K"], lens=level_lengths(cfg["K"], L, cfg["level_schedule"]),
                V=cfg["codebook_size"], d=cfg["d_lat"], dec_layers=cfg["dec_layers"], dec_heads=cfg["dec_heads"],
                dropout=cfg["tok_dropout"], ema_decay=cfg["ema_decay"], beta=cfg["beta_commit"],
                nested_p=cfg["nested_dropout"])


def gen_cfg(cfg):
    t = tok_cfg(cfg)
    return dict(K=t["K"], lens=t["lens"], V=t["V"], d_lat=t["d"], D=cfg["gen_d"], layers=cfg["gen_layers"],
                heads=cfg["gen_heads"], dropout=cfg["gen_dropout"], P=cfg["max_prompt"])


# ----------------------------------------------------------------------------- transformer parts
class Attn(nn.Module):
    def __init__(self, d, h, drop, causal=False):
        super().__init__()
        assert d % h == 0
        self.h, self.drop_p, self.causal = h, drop, causal
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)

    def forward(self, x, mask=None, cache=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        dp = self.drop_p if self.training else 0.0
        if cache is not None:                      # KV-cache path (AR baseline inference)
            if "k" in cache:                       # decode step (T == 1)
                k = torch.cat([cache["k"], k], 2); v = torch.cat([cache["v"], v], 2)
                y = F.scaled_dot_product_attention(q, k, v)
            else:                                  # prefill
                y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            cache["k"], cache["v"] = k, v
        elif mask is not None:                     # block-diagonal / prefix-LM mask (NCM)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dp, is_causal=self.causal)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, d, h, drop, causal=False):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = Attn(d, h, drop, causal)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(drop))
        self.drop = nn.Dropout(drop)

    def forward(self, x, mask=None, cache=None):
        x = x + self.drop(self.attn(self.ln1(x), mask, cache))
        return x + self.mlp(self.ln2(x))


def _init(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, 0.0, 0.02)


# ----------------------------------------------------------------------------- stage 1: multiscale residual VQ-VAE
class CodebookEMA(nn.Module):
    """VQ codebook updated by EMA (paper Table 5 lists 'Codebook EMA decay'). Data-init + dead-code
    restart are my additions (paper does not mention them) to keep utilization high."""
    def __init__(self, V, d, decay, restart_thresh=0.01, restart_every=50):
        super().__init__()
        self.V, self.d, self.decay = V, d, decay
        self.restart_thresh, self.restart_every, self._calls = restart_thresh, restart_every, 0
        self.register_buffer("embed", torch.zeros(V, d))
        self.register_buffer("cluster_size", torch.zeros(V))
        self.register_buffer("embed_avg", torch.zeros(V, d))
        self.register_buffer("inited", torch.zeros(()))

    @torch.no_grad()
    def init_from(self, u):
        flat = u.reshape(-1, self.d).float()
        n = flat.size(0)
        pick = torch.randperm(n, device=flat.device)[: self.V]
        if pick.numel() < self.V:
            pick = torch.cat([pick, torch.randint(0, n, (self.V - pick.numel(),), device=flat.device)])
        self.embed.copy_(flat[pick]); self.embed_avg.copy_(self.embed)
        self.cluster_size.fill_(1.0); self.inited.fill_(1.0)

    @torch.no_grad()
    def lookup(self, u):
        B, l, d = u.shape
        flat = u.reshape(-1, d).float()
        dist = flat.pow(2).sum(1, keepdim=True) - 2 * flat @ self.embed.t() + self.embed.pow(2).sum(1)[None]
        return dist.argmin(1).view(B, l)

    @torch.no_grad()
    def update(self, u, idx):
        flat = u.reshape(-1, self.d).float(); i = idx.reshape(-1)
        counts = torch.bincount(i, minlength=self.V).float()
        sums = torch.zeros(self.V, self.d, device=flat.device).index_add_(0, i, flat)
        self.cluster_size.mul_(self.decay).add_(counts, alpha=1 - self.decay)
        self.embed_avg.mul_(self.decay).add_(sums, alpha=1 - self.decay)
        n = self.cluster_size.sum()
        cs = (self.cluster_size + 1e-5) / (n + self.V * 1e-5) * n
        self.embed.copy_(self.embed_avg / cs[:, None])
        self._calls += 1
        if self._calls % self.restart_every == 0:
            dead = self.cluster_size < self.restart_thresh
            nd = int(dead.sum())
            if nd > 0:
                pick = torch.randint(0, flat.size(0), (nd,), device=flat.device)
                self.embed[dead] = flat[pick]; self.embed_avg[dead] = flat[pick]; self.cluster_size[dead] = 1.0


class MSRVQ(nn.Module):
    """Multiscale residual VQ-VAE tokenizer (paper Sec. 3.3).
    encoder = token embedding (+pos), 0 transformer blocks (Table 5: 'Encoder blocks 0');
    level k: u_k = Down_k(z - zhat^{k-1}) -> nearest code -> R_k(e_k(r_k)) added to zhat;
    decoder = small transformer (Table 5: 1 block for TinyStories) -> token logits."""
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.L, self.K, self.lens, self.d, self.V = c["L"], c["K"], c["lens"], c["d"], c["V"]
        self.tok_emb = nn.Embedding(c["vocab"], self.d)
        nn.init.normal_(self.tok_emb.weight, 0.0, 1.0)
        self.enc_pos = nn.Parameter(torch.randn(self.L, self.d) * 0.1)
        self.cbs = nn.ModuleList([CodebookEMA(self.V, self.d, c["ema_decay"]) for _ in range(self.K)])
        self.R = nn.ModuleList([nn.Conv1d(self.d, self.d, 3, padding=1) for _ in range(self.K)])
        for r in self.R:                           # start R_k as (upsample ∘ identity)
            nn.init.zeros_(r.weight); nn.init.zeros_(r.bias)
            with torch.no_grad():
                r.weight[:, :, 1] = torch.eye(self.d)
        self.dec_pos = nn.Parameter(torch.randn(self.L, self.d) * 0.02)
        self.drop = nn.Dropout(c["dropout"])
        self.dec_blocks = nn.ModuleList([Block(self.d, c["dec_heads"], c["dropout"]) for _ in range(c["dec_layers"])])
        self.ln_f = nn.LayerNorm(self.d)
        self.head = nn.Linear(self.d, c["vocab"], bias=False)
        self.dec_blocks.apply(_init); self.head.apply(_init)

    # --- operators
    def pool(self, x, l):                          # Down_k / generator "resize": [B,L,d] -> [B,l,d]
        return F.adaptive_avg_pool1d(x.transpose(1, 2), l).transpose(1, 2)

    def R_apply(self, k, e):                       # R_k: [B,l,d] -> [B,L,d] (linear interp + conv; conv is my assumption)
        x = e.transpose(1, 2)
        if x.size(-1) != self.L:
            x = F.interpolate(x, size=self.L, mode="linear", align_corners=False)
        return self.R[k](x).transpose(1, 2)

    def contribution(self, k, idx):
        return self.R_apply(k, self.cbs[k].embed[idx]).float()

    def encode_z(self, x):
        return self.tok_emb(x) + self.enc_pos

    def decode(self, zhat):
        h = self.drop(zhat + self.dec_pos)
        for b in self.dec_blocks:
            h = b(h)
        return self.head(self.ln_f(h))

    # --- training forward (Eq. 7-9)
    def forward_train(self, x):
        B, V = x.size(0), self.c["vocab"]
        z = self.encode_z(x).float()
        p = self.c["nested_p"]
        if p > 0:                                   # nested dropout: keep first `cut` levels (per-sample cut: assumption)
            drop = torch.rand(B, device=x.device) < p
            cut = torch.where(drop, torch.randint(1, self.K, (B,), device=x.device), torch.full((B,), self.K, device=x.device))
        else:
            cut = torch.full((B,), self.K, device=x.device)
        zfull = torch.zeros_like(z); zcut = torch.zeros_like(z)
        for k in range(self.K):
            with torch.autocast(device_type=x.device.type, enabled=False):
                u = self.pool(z - zfull, self.lens[k])            # residual detached via zfull; grad -> z (straight-through)
                ud = u.detach()
                if self.training and self.cbs[k].inited.item() == 0:
                    self.cbs[k].init_from(ud)
                idx = self.cbs[k].lookup(ud)
                e = self.cbs[k].embed[idx]
                if self.training:
                    self.cbs[k].update(ud, idx)
                e_st = u + (e - u).detach()                        # straight-through estimator
            c = self.R_apply(k, e_st).float()
            zfull = zfull + c.detach()
            zcut = zcut + (k < cut).float()[:, None, None] * c
        logits = self.decode(zcut)
        ce = F.cross_entropy(logits.float().reshape(-1, V), x.reshape(-1))
        commit = self.c["beta"] * F.mse_loss(z, zcut.detach())
        lat = F.mse_loss(z.detach(), zcut)
        return ce + commit + lat, dict(ce=ce.detach(), commit=commit.detach(), lat=lat.detach())

    # --- deterministic encoding (nearest code per level, full hierarchy)
    @torch.no_grad()
    def encode_codes(self, x):
        z = self.encode_z(x).float()
        zfull = torch.zeros_like(z); codes = []
        for k in range(self.K):
            idx = self.cbs[k].lookup(self.pool(z - zfull, self.lens[k]))
            codes.append(idx); zfull = zfull + self.contribution(k, idx)
        return codes

    @torch.no_grad()
    def level_inputs(self, codes):
        """Teacher-forced generator inputs: resize(zhat^{k-1}) to l_k positions, for all k at once (Alg. 1)."""
        B = codes[0].size(0)
        zhat = torch.zeros(B, self.L, self.d, device=codes[0].device)
        outs = []
        for k in range(self.K):
            outs.append(self.pool(zhat, self.lens[k]))
            zhat = zhat + self.contribution(k, codes[k])
        return torch.cat(outs, 1), zhat


@torch.no_grad()
def tok_eval(model, x_all, dev, bs=256):
    """Cumulative reconstruction accuracy after each level + full-hierarchy token CE (nats/token)."""
    model.eval()
    correct = torch.zeros(model.K, device=dev); total = 0; ce = 0.0
    for i in range(0, len(x_all), bs):
        x = x_all[i:i + bs]
        with amp_ctx(dev):
            codes = model.encode_codes(x); zhat = 0
            for k in range(model.K):
                zhat = zhat + model.contribution(k, codes[k])
                lg = model.decode(zhat)
                correct[k] += (lg.argmax(-1) == x).sum()
        ce += F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), x.reshape(-1), reduction="sum").item()
        total += x.numel()
    return dict(acc=(correct / total).tolist(), ce=ce / total)


@torch.no_grad()
def codebook_stats(model, x_all, dev, bs=256):
    """Per level: % of codebook entries used at least once, and code-distribution perplexity as % of V
    (the two quantities plotted in paper Fig. 16)."""
    model.eval()
    counts = [torch.zeros(model.V, device=dev) for _ in range(model.K)]
    for i in range(0, len(x_all), bs):
        with amp_ctx(dev):
            codes = model.encode_codes(x_all[i:i + bs])
        for k in range(model.K):
            counts[k] += torch.bincount(codes[k].reshape(-1), minlength=model.V).float()
    used, perp, nsamp = [], [], []
    for k in range(model.K):
        c = counts[k]; p = c / c.sum()
        H = -(p[p > 0] * p[p > 0].log()).sum().item()
        used.append(100.0 * (c > 0).float().mean().item()); perp.append(100.0 * math.exp(H) / model.V)
        nsamp.append(int(c.sum().item()))
    return dict(used_pct=used, perplexity_pct=perp, n_codes_seen=nsamp)


def load_tokenizer(path, dev):
    ck = tload(path, "cpu")
    m = MSRVQ(ck["cfg"]); m.load_state_dict(ck["model"]); m.to(dev).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m


# ----------------------------------------------------------------------------- stage 2: NCM generator
class Generator(nn.Module):
    """One shared transformer over the concatenation [prefix | level-1 block | ... | level-K block].
    Training: all K levels in one forward pass, block-diagonal attention among level blocks (paper Fig. 11d),
    plus prefix-LM attention to a prompt (paper Sec. 3.4 'Conditional generation').
    Assumptions (paper does not specify): learned slot embeddings instead of RoPE, per-level output heads."""
    def __init__(self, g):
        super().__init__()
        self.g = g
        self.K, self.lens, self.V, self.P = g["K"], g["lens"], g["V"], g["P"]
        self.offs = [0] + [int(x) for x in np.cumsum(self.lens)]
        self.N = self.offs[-1]
        D = g["D"]
        self.in_proj = nn.Linear(g["d_lat"], D)
        self.prompt_proj = nn.Linear(g["d_lat"], D)
        self.slot = nn.Parameter(torch.randn(self.N, D) * 0.02)
        self.null = nn.Parameter(torch.randn(D) * 0.02)               # null prefix token (paper: P=1 null for unconditional)
        self.prompt_pos = nn.Parameter(torch.randn(self.P, D) * 0.02)
        self.drop = nn.Dropout(g["dropout"])
        self.blocks = nn.ModuleList([Block(D, g["heads"], g["dropout"]) for _ in range(g["layers"])])
        self.ln_f = nn.LayerNorm(D)
        self.head_w = nn.Parameter(torch.randn(self.K, D, self.V) * 0.02)
        self.head_b = nn.Parameter(torch.zeros(self.K, self.V))
        bd = torch.zeros(self.N, self.N, dtype=torch.bool)
        for k in range(self.K):
            bd[self.offs[k]:self.offs[k + 1], self.offs[k]:self.offs[k + 1]] = True
        self.register_buffer("bd", bd, persistent=False)
        self.in_proj.apply(_init); self.prompt_proj.apply(_init); self.blocks.apply(_init)

    def forward(self, inputs, prompt_emb, valid):
        """inputs [B,N,d_lat] (resized cumulative reconstructions); prompt_emb [B,P,d_lat] right-aligned;
        valid [B,P] bool marks real prompt tokens. Returns list of K logits [B,l_k,V]."""
        B, P, N = inputs.size(0), self.P, self.N
        Pp = 1 + P
        hb = self.in_proj(inputs) + self.slot[None]
        hp = torch.cat([self.null.view(1, 1, -1).expand(B, 1, -1), self.prompt_proj(prompt_emb) + self.prompt_pos[None]], 1)
        h = torch.cat([hp.float(), hb.float()], 1)
        S = Pp + N; padn = (-S) % 8                                    # pad so SDPA mem-efficient kernel gets aligned masks
        vf = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=h.device), valid], 1)
        mask = torch.zeros(B, S + padn, S + padn, dtype=torch.bool, device=h.device)
        mask[:, :Pp, :Pp] = vf[:, None, :]                             # prompt <-> prompt (bidirectional)
        mask[:, Pp:S, :Pp] = vf[:, None, :]                            # each level block sees the prompt
        mask[:, Pp:S, Pp:S] = self.bd[None]                            # ...and only itself (block-diagonal)
        if padn:
            mask[:, S:, 0] = True
            h = F.pad(h, (0, 0, 0, padn))
        mask = mask[:, None]
        h = self.drop(h)
        for blk in self.blocks:
            h = blk(h, mask)
        h = self.ln_f(h)[:, Pp:S]
        return [h[:, self.offs[k]:self.offs[k + 1]] @ self.head_w[k] + self.head_b[k] for k in range(self.K)]

    def forward_level(self, k, inp, prompt_emb=None):
        """Sampling step: only the l_k positions of level k (+ prompt) are processed."""
        B = inp.size(0)
        hb = self.in_proj(inp) + self.slot[self.offs[k]:self.offs[k + 1]][None]
        hp = self.null.view(1, 1, -1).expand(B, 1, -1)
        if prompt_emb is not None and prompt_emb.size(1) > 0:
            m = prompt_emb.size(1)
            hp = torch.cat([hp, self.prompt_proj(prompt_emb) + self.prompt_pos[self.P - m:][None]], 1)
        Pp = hp.size(1); l = hb.size(1); T = Pp + l
        h = torch.cat([hp.float(), hb.float()], 1)
        mask = torch.ones(T, T, dtype=torch.bool, device=h.device)
        mask[:Pp, Pp:] = False                                         # prompt rows never look at the block
        mask = mask[None, None]
        for blk in self.blocks:
            h = blk(h, mask)
        h = self.ln_f(h)[:, Pp:]
        return h @ self.head_w[k] + self.head_b[k]


def corrupt_codes(codes, rates, V):
    """Conditioning corruption (paper Sec. 3.4/F.3): replace a fraction of INPUT codes with uniform codes from
    the same level's codebook; labels stay clean. Per-level rates decrease linearly (shape assumed)."""
    out = []
    for k, c in enumerate(codes):
        m = torch.rand(c.shape, device=c.device) < rates[k]
        out.append(torch.where(m, torch.randint_like(c, V), c))
    return out


@torch.no_grad()
def gen_eval(gen, tok, wins, m, dev, recon_nll, bs=64):
    """Teacher-forced, clean-conditioning eval. Returns mean CE per code, per-level CE, and the paper's ELBO
    bound (App. A.2) converted to nats/token: (sum_k sum_i CE_ki)/L + recon_nll."""
    gen.eval(); P, L, K = gen.P, tok.L, gen.K
    sums = torch.zeros(K, device=dev); n = 0
    for i in range(0, len(wins), bs):
        w = wins[i:i + bs]; B = w.size(0)
        ctx, tgt = w[:, :P], w[:, P:]
        valid = (torch.arange(P, device=dev)[None] >= (P - m)).expand(B, P)
        with amp_ctx(dev):
            codes = tok.encode_codes(tgt)
            inputs, _ = tok.level_inputs(codes)
            logits = gen(inputs, tok.tok_emb(ctx), valid)
        for k in range(K):
            sums[k] += F.cross_entropy(logits[k].float().reshape(-1, gen.V), codes[k].reshape(-1), reduction="sum")
        n += B
    lens = torch.tensor(gen.lens, device=dev, dtype=torch.float32)
    per_level = (sums / (n * lens)).tolist()
    return dict(ce=(sums.sum() / (n * gen.N)).item(), per_level=per_level,
                bound_nats_per_tok=(sums.sum() / (n * L)).item() + recon_nll)


@torch.no_grad()
def tok_recon_nll(tok, x_all, dev):
    return tok_eval(tok, x_all, dev)["ce"]


def load_generator(path, dev):
    ck = tload(path, "cpu")
    g = Generator(ck["cfg"]); g.load_state_dict(ck["model"]); return g.to(dev).eval()


# ----------------------------------------------------------------------------- nanoGPT-style baseline
class GPT(nn.Module):
    """nanoGPT-style: learned pos-emb, pre-LN blocks, GELU MLP, tied lm_head. Extra id `vocab` = BOS so that the
    first token of a random window has a proper unconditional distribution."""
    def __init__(self, vocab, block, n_layer, d, heads, dropout):
        super().__init__()
        self.vocab, self.block, self.bos = vocab, block, vocab
        self.cfg = dict(vocab=vocab, block=block, n_layer=n_layer, d=d, heads=heads, dropout=dropout)
        self.wte = nn.Embedding(vocab + 1, d); self.wpe = nn.Embedding(block, d)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d, heads, dropout, causal=True) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab + 1, bias=False)
        self.lm_head.weight = self.wte.weight
        self.apply(_init)

    def forward(self, idx):
        T = idx.size(1)
        x = self.drop(self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))[None])
        for b in self.blocks:
            x = b(x)
        return self.lm_head(self.ln_f(x))

    def forward_cached(self, idx, caches, start):
        T = idx.size(1)
        x = self.wte(idx) + self.wpe(torch.arange(start, start + T, device=idx.device))[None]
        for b, c in zip(self.blocks, caches):
            x = b(x, cache=c)
        return self.lm_head(self.ln_f(x[:, -1]))

    def _sample(self, logits, temperature):
        logits = logits.float().clone(); logits[:, self.bos] = float("-inf")
        if temperature <= 0:
            return logits.argmax(-1)
        return torch.multinomial(F.softmax(logits / temperature, -1), 1)[:, 0]

    @torch.no_grad()
    def generate(self, prefix, n_new, temperature=1.0, kv_cache=True):
        new = []
        if kv_cache:
            assert prefix.size(1) + n_new - 1 <= self.block
            caches = [dict() for _ in self.blocks]
            logits = self.forward_cached(prefix, caches, 0); pos = prefix.size(1)
            for i in range(n_new):
                nxt = self._sample(logits, temperature); new.append(nxt)
                if i < n_new - 1:
                    logits = self.forward_cached(nxt[:, None], caches, pos); pos += 1
        else:                                       # classic nanoGPT loop: recompute the whole context every step
            idx = prefix
            for i in range(n_new):
                nxt = self._sample(self(idx[:, -self.block:])[:, -1], temperature); new.append(nxt)
                idx = torch.cat([idx, nxt[:, None]], 1)
        return torch.stack(new, 1)


def load_gpt(path, dev):
    ck = tload(path, "cpu")
    m = GPT(**ck["cfg"]); m.load_state_dict(ck["model"]); return m.to(dev).eval()


def match_ar_layers(cfg, target_params):
    d, best = cfg["gen_d"], None
    for nl in range(1, 33):
        n = count_params(GPT(cfg["vocab"], cfg["max_prompt"] + cfg["block_len"], nl, d, cfg["gen_heads"], 0.0))
        if best is None or abs(n - target_params) < abs(best[1] - target_params):
            best = (nl, n)
    return best[0]


# ----------------------------------------------------------------------------- sampling / metrics
def sample_logits(logits, temperature):
    if temperature <= 0:
        return logits.argmax(-1)
    B, l, V = logits.shape
    return torch.multinomial(F.softmax(logits.float() / temperature, -1).reshape(-1, V), 1).view(B, l)


@torch.no_grad()
def ncm_sample(tok, gen, n, bs, dev, temperature=1.0, prompt_ids=None, seed=None, traj=False):
    """Algorithm 2: K sequential steps; each predicts all l_k codes of a level in parallel. Tokens by argmax."""
    if seed is not None:
        torch.manual_seed(seed)
    outs, trajs = [], []
    for i in range(0, n, bs):
        b = min(bs, n - i)
        zhat = torch.zeros(b, tok.L, tok.d, device=dev)
        pe = None
        if prompt_ids is not None and prompt_ids.numel() > 0:
            pe = tok.tok_emb(prompt_ids.view(1, -1).expand(b, -1))
        tr = []
        with amp_ctx(dev):
            for k in range(tok.K):
                logits = gen.forward_level(k, tok.pool(zhat, tok.lens[k]), pe)
                zhat = zhat + tok.contribution(k, sample_logits(logits, temperature))
                if traj:
                    tr.append(tok.decode(zhat).argmax(-1))
            outs.append(tok.decode(zhat).argmax(-1))
        if traj:
            trajs.append(torch.stack(tr, 0))
    x = torch.cat(outs, 0)
    return (x, torch.cat(trajs, 1)) if traj else x


@torch.no_grad()
def ar_sample(gpt, n, bs, dev, n_new, temperature=1.0, prompt_ids=None, kv_cache=True, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    outs = []
    for i in range(0, n, bs):
        b = min(bs, n - i)
        prefix = torch.full((b, 1), gpt.bos, dtype=torch.long, device=dev)
        if prompt_ids is not None and prompt_ids.numel() > 0:
            prefix = torch.cat([prefix, prompt_ids.view(1, -1).expand(b, -1)], 1)
        with amp_ctx(dev):
            outs.append(gpt.generate(prefix, n_new, temperature, kv_cache))
    return torch.cat(outs, 0)


def unigram_entropy(ids):
    """Mean per-sample unigram entropy (nats) of the model's own BPE token ids (paper reports mean per-sample H)."""
    ids = ids.cpu().numpy(); H = []
    for row in ids:
        _, c = np.unique(row, return_counts=True); p = c / c.sum()
        H.append(float(-(p * np.log(p)).sum()))
    return float(np.mean(H))


def load_ref_model(name_or_path, dev):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tk = AutoTokenizer.from_pretrained(name_or_path)
    m = AutoModelForCausalLM.from_pretrained(name_or_path, torch_dtype=torch.float16 if dev.type == "cuda" else torch.float32)
    return m.to(dev).eval(), tk


@torch.no_grad()
def gen_ppl(texts, ref_model, encode_fn, bos_id, dev, bs=16, max_len=256):
    """Generative perplexity under a reference LM: exp(total NLL / total scored tokens), BOS-prefixed so the
    first token is scored. (Paper uses GPT-2-large; pooling choice here is my assumption.)"""
    nll_sum, n_tok = 0.0, 0
    for i in range(0, len(texts), bs):
        enc = [[bos_id] + encode_fn(t)[: max_len - 1] for t in texts[i:i + bs]]
        ml = max(len(e) for e in enc)
        ids = torch.tensor([e + [bos_id] * (ml - len(e)) for e in enc], device=dev)
        att = torch.tensor([[1] * len(e) + [0] * (ml - len(e)) for e in enc], device=dev)
        lg = ref_model(input_ids=ids, attention_mask=att).logits.float()
        lp = F.log_softmax(lg[:, :-1], -1).gather(-1, ids[:, 1:, None])[..., 0]
        m = att[:, 1:].float()
        nll_sum += float(-(lp * m).sum()); n_tok += float(m.sum())
    return math.exp(nll_sum / max(n_tok, 1))


# ----------------------------------------------------------------------------- training loops
def _ckpt_state(model, opt, scaler, state):
    return dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), state=state)


def run_tok(cfg, args):
    dev = pick_device(cfg); set_seed(cfg["seed"])
    work = Path(cfg["work"]); out = work / "tok"; out.mkdir(parents=True, exist_ok=True)
    data = TextData(work / "data", dev)
    tc = tok_cfg(cfg); L = tc["L"]
    model = MSRVQ(tc).to(dev)
    print(f"[tok] params={count_params(model)/1e6:.2f}M  lens={tc['lens']}  device={dev}", flush=True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["tok_lr"], betas=tuple(cfg["tok_betas"]), weight_decay=cfg["tok_wd"])
    scaler = make_scaler(dev)
    state = dict(step=0, hist=[], best=1e9, elapsed=0.0, done=False)
    if (out / "last.pt").exists():
        ck = tload(out / "last.pt", dev)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); state = ck["state"]
        print(f"[tok] RESUMED from step {state['step']}", flush=True)
    for k in range(model.K):
        model.cbs[k]._calls = state["step"]
    val_x = data.val_windows(L, cfg["val_stride"])
    total = cfg["tok_steps"]; t0 = time.time() - state["elapsed"]; model.train(); run_loss = 0.0; nl = 0
    while state["step"] < total and not state["done"]:
        step = state["step"]
        for gq in opt.param_groups:
            gq["lr"] = lr_at(step, total, cfg["tok_warmup"], cfg["tok_lr"], cfg["tok_lr_floor"])
        x = data.train_windows(cfg["tok_batch"], L)
        with amp_ctx(dev):
            loss, info = model.forward_train(x)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        state["step"] += 1; run_loss += loss.item() if step % 10 == 0 else 0; nl += step % 10 == 0
        elapsed = time.time() - t0
        if state["step"] % cfg["log_every"] == 0:
            print(f"[tok] step {state['step']} loss {loss.item():.4f} ce {info['ce'].item():.4f} lat {info['lat'].item():.4f} "
                  f"lr {opt.param_groups[0]['lr']:.2e} elapsed {elapsed/60:.1f}m", flush=True)
        timeup = elapsed > cfg["tok_max_minutes"] * 60
        stop_now = args.stop_after and state["step"] >= args.stop_after
        if state["step"] % cfg["tok_eval_every"] == 0 or state["step"] == total or timeup or stop_now:
            ev = tok_eval(model, val_x, dev); model.train()
            state["hist"].append(dict(step=state["step"], train_loss=run_loss / max(nl, 1), val_ce=ev["ce"], val_acc_full=ev["acc"][-1]))
            run_loss, nl = 0.0, 0
            print(f"[tok] EVAL step {state['step']} val_ce {ev['ce']:.4f} val_acc(full) {ev['acc'][-1]*100:.2f}%", flush=True)
            if ev["ce"] < state["best"]:
                state["best"] = ev["ce"]; save_atomic(dict(model=model.state_dict(), cfg=tc), out / "best.pt")
            jdump(state["hist"], out / "history.json")
        if state["step"] % cfg["ckpt_every"] == 0 or timeup or stop_now:
            state["elapsed"] = time.time() - t0
            save_atomic(_ckpt_state(model, opt, scaler, state), out / "last.pt")
        if stop_now:
            print(f"[tok] stop_after reached at step {state['step']} (simulated interruption)", flush=True); return
        if timeup:
            print("[tok] time budget reached", flush=True); break
    state["done"] = True; state["elapsed"] = time.time() - t0
    save_atomic(_ckpt_state(model, opt, scaler, state), out / "last.pt")
    best = tload(out / "best.pt", dev); model.load_state_dict(best["model"]); model.eval()
    save_atomic(dict(model=model.state_dict(), cfg=tc), out / "tokenizer.pt")
    g = torch.Generator().manual_seed(7)
    tr_x = data.train_windows(cfg["util_train_windows"], L, gen=g)
    vs_x = data.val_windows(L, cfg["util_val_stride"])
    res = dict(params=count_params(model), lens=tc["lens"],
               train=dict(**tok_eval(model, tr_x, dev), **codebook_stats(model, tr_x, dev), n_windows=len(tr_x)),
               val=dict(**tok_eval(model, vs_x, dev), **codebook_stats(model, vs_x, dev), n_windows=len(vs_x)))
    jdump(res, out / "tok_metrics.json")
    print(f"[tok] DONE  val recon acc (full hierarchy) {res['val']['acc'][-1]*100:.2f}%", flush=True)


def run_gen(cfg, args):
    dev = pick_device(cfg); set_seed(cfg["seed"])
    work = Path(cfg["work"]); out = work / "gen"; out.mkdir(parents=True, exist_ok=True)
    tok = load_tokenizer(work / "tok" / "tokenizer.pt", dev); data = TextData(work / "data", dev)
    L, P, K = cfg["block_len"], cfg["max_prompt"], cfg["K"]
    gc = gen_cfg(cfg); gen = Generator(gc).to(dev)
    print(f"[gen] params={count_params(gen)/1e6:.2f}M (frozen tokenizer {count_params(tok)/1e6:.2f}M)  N codes/sample={gen.N}", flush=True)
    decay = [p for n, p in gen.named_parameters() if p.dim() >= 2 and not any(t in n for t in ("slot", "prompt_pos"))]
    nodecay = [p for n, p in gen.named_parameters() if not (p.dim() >= 2 and not any(t in n for t in ("slot", "prompt_pos")))]
    opt = torch.optim.AdamW([dict(params=decay, weight_decay=cfg["gen_wd"]), dict(params=nodecay, weight_decay=0.0)],
                            lr=cfg["gen_lr"], betas=tuple(cfg["gen_betas"]))
    scaler = make_scaler(dev)
    state = dict(step=0, hist=[], best=1e9, best_step=0, bad=0, elapsed=0.0, done=False, reason="")
    if (out / "last.pt").exists():
        ck = tload(out / "last.pt", dev)
        gen.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); state = ck["state"]
        print(f"[gen] RESUMED from step {state['step']}", flush=True)
        if state["done"]:
            print("[gen] already finished", flush=True); return
    val_w = data.val_windows(P + L, cfg["val_stride"])
    tr_w = data.train_windows(cfg["eval_train_windows"], P + L, gen=torch.Generator().manual_seed(123))
    recon = tok_recon_nll(tok, val_w[:, P:], dev)
    print(f"[gen] val windows={len(val_w)}  tokenizer recon NLL on val={recon:.4f} nats/tok", flush=True)
    rates = torch.linspace(cfg["corrupt_hi"], cfg["corrupt_lo"], K, device=dev)
    lens_t = torch.tensor(gc["lens"], dtype=torch.float32, device=dev)
    wk = lens_t.pow(-cfg["alpha"]) / lens_t.pow(1 - cfg["alpha"]).sum()          # Eq. 10 weights
    total = cfg["gen_steps"]; t0 = time.time() - state["elapsed"]; B = cfg["gen_batch"]
    run_loss, nl, run_lvl = 0.0, 0, torch.zeros(K, device=dev); gen.train()
    while state["step"] < total:
        step = state["step"]
        for gq in opt.param_groups:
            gq["lr"] = lr_at(step, total, cfg["gen_warmup"], cfg["gen_lr"], cfg["gen_lr_floor"])
        w = data.train_windows(B, P + L); ctx, tgt = w[:, :P], w[:, P:]
        use = torch.rand(B, device=dev) < cfg["cond_prob"]
        m = torch.where(use, torch.randint(1, P + 1, (B,), device=dev), torch.zeros(B, dtype=torch.long, device=dev))
        valid = torch.arange(P, device=dev)[None] >= (P - m)[:, None]
        with torch.no_grad(), amp_ctx(dev):
            codes = tok.encode_codes(tgt)
            inputs, _ = tok.level_inputs(corrupt_codes(codes, rates, gc["V"]))
            pe = tok.tok_emb(ctx)
        with amp_ctx(dev):
            logits = gen(inputs, pe, valid)
        lvl = torch.stack([F.cross_entropy(logits[k].float().reshape(-1, gen.V), codes[k].reshape(-1), reduction="sum") / B for k in range(K)])
        loss = (wk * lvl).sum()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        state["step"] += 1
        if step % 10 == 0:
            run_loss += (lvl.detach().sum() / gen.N).item(); nl += 1
        run_lvl += lvl.detach() / lens_t
        elapsed = time.time() - t0
        if state["step"] % cfg["log_every"] == 0:
            print(f"[gen] step {state['step']} train_ce/code {(lvl.detach().sum()/gen.N).item():.4f} lr {opt.param_groups[0]['lr']:.2e} "
                  f"elapsed {elapsed/60:.1f}m", flush=True)
        timeup = elapsed > cfg["gen_max_minutes"] * 60
        stop_now = args.stop_after and state["step"] >= args.stop_after
        if state["step"] % cfg["eval_every"] == 0 or state["step"] == total or timeup or stop_now:
            ev = gen_eval(gen, tok, val_w, 0, dev, recon); evc = gen_eval(gen, tok, val_w, P, dev, recon)
            etr = gen_eval(gen, tok, tr_w, 0, dev, recon); gen.train()
            rec = dict(step=state["step"], train_ce_corrupt=run_loss / max(nl, 1), train_ce_clean=etr["ce"], val_ce=ev["ce"],
                       val_ce_prompted=evc["ce"], val_bound=ev["bound_nats_per_tok"], train_bound=etr["bound_nats_per_tok"],
                       val_per_level=ev["per_level"], elapsed_min=elapsed / 60)
            state["hist"].append(rec); run_loss, nl = 0.0, 0
            print(f"[gen] EVAL step {state['step']} val_ce {ev['ce']:.4f} train_clean_ce {etr['ce']:.4f} "
                  f"val_prompted_ce {evc['ce']:.4f} val_bound {ev['bound_nats_per_tok']:.3f} nats/tok", flush=True)
            if ev["ce"] < state["best"] - 1e-4:
                state.update(best=ev["ce"], best_step=state["step"], bad=0)
                save_atomic(dict(model=gen.state_dict(), cfg=gc, step=state["step"]), out / "best.pt")
            else:
                state["bad"] += 1
            jdump(state["hist"], out / "history.json")
            if state["bad"] >= cfg["patience"]:
                state["reason"] = f"early stopping (no val improvement for {cfg['patience']} evals; best step {state['best_step']})"
                print(f"[gen] {state['reason']}", flush=True); break
        if state["step"] % cfg["ckpt_every"] == 0 or timeup or stop_now:
            state["elapsed"] = time.time() - t0
            save_atomic(_ckpt_state(gen, opt, scaler, state), out / "last.pt")
            print(f"[gen] checkpoint saved at step {state['step']}", flush=True)
        if stop_now:
            print(f"[gen] stop_after reached at step {state['step']} (simulated interruption)", flush=True); return
        if timeup:
            state["reason"] = "time budget"; print("[gen] time budget reached", flush=True); break
    if not state["reason"]:
        state["reason"] = "max steps"
    state["done"] = True; state["elapsed"] = time.time() - t0
    save_atomic(_ckpt_state(gen, opt, scaler, state), out / "last.pt")
    if not (out / "best.pt").exists():
        save_atomic(dict(model=gen.state_dict(), cfg=gc, step=state["step"]), out / "best.pt")
    save_atomic(dict(model=tload(out / "best.pt", "cpu")["model"], cfg=gc), out / "generator.pt")
    jdump(dict(params=count_params(gen), best_step=state["best_step"], best_val_ce=state["best"], reason=state["reason"],
               train_minutes=state["elapsed"] / 60, tokenizer_recon_nll_val=recon), out / "final.json")
    print(f"[gen] DONE: {state['reason']}", flush=True)


def run_ar(cfg, args):
    dev = pick_device(cfg); set_seed(cfg["seed"])
    work = Path(cfg["work"]); out = work / "ar"; out.mkdir(parents=True, exist_ok=True)
    data = TextData(work / "data", dev)
    L, P = cfg["block_len"], cfg["max_prompt"]
    target = count_params(Generator(gen_cfg(cfg)))
    nl = cfg["ar_layers"] or match_ar_layers(cfg, target)
    gpt = GPT(cfg["vocab"], P + L, nl, cfg["gen_d"], cfg["gen_heads"], cfg["ar_dropout"]).to(dev)
    print(f"[ar] NCM generator params={target/1e6:.2f}M -> AR n_layer={nl}, params={count_params(gpt)/1e6:.2f}M", flush=True)
    jdump(dict(ncm_generator_params=target, ar_params=count_params(gpt), n_layer=nl), out / "match.json")
    dec = [p for p in gpt.parameters() if p.dim() >= 2]; nod = [p for p in gpt.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([dict(params=dec, weight_decay=cfg["ar_wd"]), dict(params=nod, weight_decay=0.0)],
                            lr=cfg["ar_lr"], betas=tuple(cfg["gen_betas"]))
    scaler = make_scaler(dev)
    state = dict(step=0, hist=[], best=1e9, best_step=0, bad=0, elapsed=0.0, done=False, reason="")
    if (out / "last.pt").exists():
        ck = tload(out / "last.pt", dev)
        gpt.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); state = ck["state"]
        print(f"[ar] RESUMED from step {state['step']}", flush=True)
        if state["done"]:
            return
    val_w = data.val_windows(P + L, cfg["val_stride"])
    tr_w = data.train_windows(cfg["eval_train_windows"], P + L, gen=torch.Generator().manual_seed(123))

    @torch.no_grad()
    def nll(wins):                                  # unconditional: [BOS] + 128-token block; NLL per token (nats)
        gpt.eval(); s = 0.0; n = 0
        for i in range(0, len(wins), 128):
            t = wins[i:i + 128, P:]; inp = torch.cat([torch.full((t.size(0), 1), gpt.bos, device=dev), t[:, :-1]], 1)
            with amp_ctx(dev):
                lg = gpt(inp)
            s += F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), t.reshape(-1), reduction="sum").item(); n += t.numel()
        gpt.train(); return s / n

    total = cfg["ar_steps"]; t0 = time.time() - state["elapsed"]; B = cfg["ar_batch"]; run, nr = 0.0, 0; gpt.train()
    while state["step"] < total:
        step = state["step"]
        for gq in opt.param_groups:
            gq["lr"] = lr_at(step, total, cfg["ar_warmup"], cfg["ar_lr"], cfg["ar_lr_floor"])
        w = data.train_windows(B, P + L)
        inp = torch.cat([torch.full((B, 1), gpt.bos, device=dev), w[:, :-1]], 1)
        with amp_ctx(dev):
            lg = gpt(inp)
        loss = F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), w.reshape(-1))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        state["step"] += 1
        if step % 10 == 0:
            run += loss.item(); nr += 1
        elapsed = time.time() - t0
        if state["step"] % cfg["log_every"] == 0:
            print(f"[ar] step {state['step']} loss {loss.item():.4f} lr {opt.param_groups[0]['lr']:.2e} elapsed {elapsed/60:.1f}m", flush=True)
        timeup = elapsed > cfg["ar_max_minutes"] * 60
        stop_now = args.stop_after and state["step"] >= args.stop_after
        if state["step"] % cfg["ar_eval_every"] == 0 or state["step"] == total or timeup or stop_now:
            v, tr = nll(val_w), nll(tr_w)
            state["hist"].append(dict(step=state["step"], train_loss=run / max(nr, 1), train_eval_nll=tr, val_nll=v, elapsed_min=elapsed / 60)); run, nr = 0.0, 0
            print(f"[ar] EVAL step {state['step']} val_nll {v:.4f} train_nll {tr:.4f}", flush=True)
            if v < state["best"] - 1e-4:
                state.update(best=v, best_step=state["step"], bad=0); save_atomic(dict(model=gpt.state_dict(), cfg=gpt.cfg), out / "best.pt")
            else:
                state["bad"] += 1
            jdump(state["hist"], out / "history.json")
            if state["bad"] >= cfg["patience"]:
                state["reason"] = f"early stopping (best step {state['best_step']})"; print(f"[ar] {state['reason']}", flush=True); break
        if state["step"] % cfg["ckpt_every"] == 0 or timeup or stop_now:
            state["elapsed"] = time.time() - t0; save_atomic(_ckpt_state(gpt, opt, scaler, state), out / "last.pt")
        if stop_now:
            return
        if timeup:
            state["reason"] = "time budget"; break
    state["reason"] = state["reason"] or "max steps"; state["done"] = True; state["elapsed"] = time.time() - t0
    save_atomic(_ckpt_state(gpt, opt, scaler, state), out / "last.pt")
    best = tload(out / "best.pt", "cpu")
    save_atomic(dict(model=best["model"], cfg=gpt.cfg), out / "gpt.pt")
    jdump(dict(params=count_params(gpt), best_step=state["best_step"], best_val_nll=state["best"], reason=state["reason"],
               train_minutes=state["elapsed"] / 60), out / "final.json")
    print(f"[ar] DONE: {state['reason']}", flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=["tok", "gen", "ar"]); ap.add_argument("--cfg", required=True)
    ap.add_argument("--stop_after", type=int, default=0)
    args = ap.parse_args()
    cfg = json.load(open(args.cfg))
    {"tok": run_tok, "gen": run_gen, "ar": run_ar}[args.stage](cfg, args)


if __name__ == "__main__":
    main()

## 1 · Data: Tiny Shakespeare → 90/10 split → BPE

In [ ]:
import urllib.request, numpy as np
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
DATA = WORK/'data'; DATA.mkdir(exist_ok=True)
def load_text():
    for p in Path('/kaggle/input').rglob('*.txt') if Path('/kaggle/input').exists() else []:
        if 'shakespeare' in str(p).lower() or p.name == 'input.txt': return p.read_text()
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'  # same URL nanoGPT uses
    try: return urllib.request.urlopen(url, timeout=30).read().decode()
    except Exception as e: raise SystemExit(f'No internet: attach a Tiny Shakespeare dataset to the notebook ({e})')
text = load_text(); cut = int(len(text)*(1-CFG['val_frac'])); tr_txt, va_txt = text[:cut], text[cut:]   # contiguous 90/10
bpe = Tokenizer(models.BPE()); bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False); bpe.decoder = decoders.ByteLevel()
bpe.train_from_iterator([tr_txt], trainers.BpeTrainer(vocab_size=CFG['bpe_vocab'], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
assert bpe.decode(bpe.encode(tr_txt[:5000]).ids) == tr_txt[:5000]          # lossless round trip
tr_ids, va_ids = bpe.encode(tr_txt).ids, bpe.encode(va_txt).ids
np.save(DATA/'train_ids.npy', np.array(tr_ids, dtype=np.int32)); np.save(DATA/'val_ids.npy', np.array(va_ids, dtype=np.int32)); bpe.save(str(DATA/'bpe.json'))
CFG['vocab'] = bpe.get_vocab_size(); save_cfg()
print(f"chars {len(text):,} | BPE vocab {CFG['vocab']} | train tokens {len(tr_ids):,} | val tokens {len(va_ids):,} | {len(tr_txt)/len(tr_ids):.2f} chars/token")

## 2 · Stage 1 tokenizer (GPU 0) ‖ AR baseline (GPU 1) — run concurrently

In [ ]:
import torch
NGPU = torch.cuda.device_count()
def launch(stage, gpu, extra=()):
    env = os.environ.copy()
    if gpu is not None: env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.Popen([sys.executable, str(WORK/'ncm_lib.py'), stage, '--cfg', str(WORK/'config.json'), *extra],
                            stdout=open(WORK/f'{stage}.log','a'), stderr=subprocess.STDOUT, env=env, cwd=WORK)
def wait(procs, every=30):
    from IPython.display import clear_output
    while any(p.poll() is None for _, p in procs):
        clear_output(wait=True)
        for n, p in procs: print(f'--- {n} ---\n' + ''.join(open(WORK/f'{n}.log').readlines()[-3:]))
        time.sleep(every)
    for n, p in procs:
        if p.returncode: print(''.join(open(WORK/f'{n}.log').readlines()[-30:])); raise RuntimeError(f'{n} failed')
    print('done:', [n for n, _ in procs])
print('GPUs:', NGPU)
wait([('tok', launch('tok', 0 if NGPU else None)), ('ar', launch('ar', (1 if NGPU > 1 else 0) if NGPU else None))])
print(open(WORK/'ar'/'match.json').read())

### Deliverable 1 — tokenizer: reconstruction accuracy & codebook utilization per level

In [ ]:
import matplotlib.pyplot as plt
tm = json.load(open(WORK/'tok'/'tok_metrics.json')); lens = tm['lens']; K = len(lens)
print(f"tokenizer params {tm['params']/1e6:.2f}M | level lengths {lens}")
print(f"{'lvl':>3} {'len':>4} | {'recon acc tr/val %':>20} | {'used% tr/val':>14} | {'perplexity% tr/val':>20}")
for k in range(K):
    print(f"{k+1:>3} {lens[k]:>4} | {tm['train']['acc'][k]*100:9.2f} /{tm['val']['acc'][k]*100:8.2f} | "
          f"{tm['train']['used_pct'][k]:6.1f} /{tm['val']['used_pct'][k]:5.1f} | {tm['train']['perplexity_pct'][k]:9.1f} /{tm['val']['perplexity_pct'][k]:8.1f}")
print(f"NOTE: val utilization is capped by #codes seen at coarse levels (val windows={tm['val']['n_windows']}, codes seen at level 1: {tm['val']['n_codes_seen'][0]}); use the train column for coverage.")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5)); x = range(1, K+1)
for s in ('train','val'):
    ax[0].plot(x, [a*100 for a in tm[s]['acc']], 'o-', label=s); ax[1].plot(x, tm[s]['used_pct'], 'o-', label=s); ax[2].plot(x, tm[s]['perplexity_pct'], 'o-', label=s)
for a, t in zip(ax, ['cumulative recon accuracy % (levels ≤ k)', 'codebook entries used %', 'code perplexity % of V']): a.set_title(t); a.set_xlabel('level'); a.legend()
plt.tight_layout(); plt.savefig(WORK/'tok_levels.png', dpi=110); plt.show()
h = json.load(open(WORK/'tok'/'history.json')); plt.figure(figsize=(5,3)); plt.plot([r['step'] for r in h], [r['val_ce'] for r in h], 'o-'); plt.title('tokenizer val recon CE'); plt.show()

## 3 · Stage 2 generator (GPU 0): early stopping, checkpoints every `ckpt_every` steps, resume demo
First run is **interrupted on purpose** at the first checkpoint, then relaunched: it resumes from `gen/last.pt`.

In [ ]:
wait([('gen', launch('gen', 0 if NGPU else None, ['--stop_after', str(CFG['ckpt_every'])]))], every=20)   # simulated crash after first checkpoint
wait([('gen', launch('gen', 0 if NGPU else None))], every=30)                                           # resumes automatically
print([l.strip() for l in open(WORK/'gen.log') if 'RESUMED' in l or 'DONE' in l or 'early' in l or 'checkpoint' in l])
print(open(WORK/'gen'/'final.json').read())

### Deliverable 2 — generator & baseline loss curves

In [ ]:
gh = json.load(open(WORK/'gen'/'history.json')); ah = json.load(open(WORK/'ar'/'history.json')); gf = json.load(open(WORK/'gen'/'final.json')); af = json.load(open(WORK/'ar'/'final.json'))
fig, ax = plt.subplots(1, 3, figsize=(16, 3.6)); s = [r['step'] for r in gh]
ax[0].plot(s, [r['train_ce_corrupt'] for r in gh], label='train (corrupted cond.)'); ax[0].plot(s, [r['train_ce_clean'] for r in gh], label='train (clean cond.)')
ax[0].plot(s, [r['val_ce'] for r in gh], label='val'); ax[0].axvline(gf['best_step'], ls=':', c='k'); ax[0].set_title('NCM generator: CE per code (nats)'); ax[0].legend()
ax[1].plot([r['step'] for r in ah], [r['train_eval_nll'] for r in ah], label='train'); ax[1].plot([r['step'] for r in ah], [r['val_nll'] for r in ah], label='val'); ax[1].axvline(af['best_step'], ls=':', c='k'); ax[1].set_title('AR baseline: NLL / token (nats)'); ax[1].legend()
ax[2].plot(s, [r['val_bound'] for r in gh], label='NCM val ELBO bound (nats/token)'); ax[2].plot([r['step'] for r in ah], [r['val_nll'] for r in ah], label='AR val NLL (nats/token)'); ax[2].set_title('val quality, same units'); ax[2].legend()
plt.tight_layout(); plt.savefig(WORK/'curves.png', dpi=110); plt.show()
print('NCM:', gf['reason'], '| AR:', af['reason'])

## 4 · Sampling (deliverable 3)

In [ ]:
from ncm_lib import *
dev = pick_device(CFG); bpe = Tokenizer.from_file(str(DATA/'bpe.json')); P, L = CFG['max_prompt'], CFG['block_len']
tok = load_tokenizer(WORK/'tok'/'tokenizer.pt', dev); gen = load_generator(WORK/'gen'/'generator.pt', dev); gpt = load_gpt(WORK/'ar'/'gpt.pt', dev)
dec = lambda ids: bpe.decode(ids.tolist())
xs = ncm_sample(tok, gen, CFG['n_show'], 32, dev, seed=0)
print('=== NCM: unconditional samples ===')
for i, x in enumerate(xs): print(f'--- NCM #{i+1} ---\n{dec(x)}\n')
prompts = ['ROMEO:\n', 'To be, or not to be', 'First Citizen:\nBefore we proceed any further, hear me speak.']
print('=== prompt-conditioned (NCM prefix-LM  |  AR) ===')
for pr in prompts:
    pid = torch.tensor(bpe.encode(pr).ids[-P:], device=dev)
    n_out = ncm_sample(tok, gen, 2, 2, dev, prompt_ids=pid, seed=1); a_out = ar_sample(gpt, 2, 2, dev, L, prompt_ids=pid, seed=1)
    for j in range(2): print(f'### PROMPT {pr!r}\n[NCM] {dec(n_out[j])}\n[AR ] {dec(a_out[j])}\n')
xs1, tr = ncm_sample(tok, gen, 1, 1, dev, seed=3, traj=True)
print('=== coarse→fine trajectory (decode after level k) ===')
for k in [0, 2, 5, 8, tok.K-1]: print(f'-- after level {k+1} ({tok.lens[k]} codes) --\n{dec(tr[k, 0])[:300]}\n')

## 5 · Metrics (deliverable 5): generative perplexity, unigram entropy, NFE

In [ ]:
ref, ref_tk, ref_name = None, None, None
for name in ([CFG['ref_path']] if CFG['ref_path'] else CFG['ref_models']):
    try: ref, ref_tk = load_ref_model(name, dev); ref_name = name; break
    except Exception as e: print('could not load', name, '->', repr(e)[:120])
bos = (ref_tk.bos_token_id if ref_tk is not None and ref_tk.bos_token_id is not None else None)
if ref_tk is not None and bos is None: bos = ref_tk.eos_token_id
def gppl(ids): return gen_ppl([dec(r) for r in ids], ref, lambda t: ref_tk.encode(t), bos, dev) if ref is not None else float('nan')
nS = CFG['n_samples']; RES = {'ref_model': ref_name, 'rows': []}
val_blocks = TextData(DATA, dev).val_windows(L, L)
def row(name, ids, nfe):
    r = dict(name=name, gen_ppl=gppl(ids), entropy=unigram_entropy(ids), nfe=nfe); RES['rows'].append(r)
    print(f"{name:<22} GenPPL[{ref_name}] {r['gen_ppl']:8.2f} | unigram H {r['entropy']:.3f} nats | NFE/sample {nfe}")
row('data (val blocks)', val_blocks, '-')
for T in CFG['temps']:
    n = nS if T == 1.0 else nS // 2
    row(f'NCM  T={T}', ncm_sample(tok, gen, n, 64, dev, temperature=T, seed=11), tok.K)
    row(f'AR   T={T}', ar_sample(gpt, n, 64, dev, L, temperature=T, seed=11), L)
print('Reference LM is GPT-2 (web text), not Shakespeare: GenPPL is a relative signal only. Low PPL with low entropy = repetition, not quality.')
del ref; torch.cuda.empty_cache() if dev.type == 'cuda' else None

## 6 · Benchmark (deliverable 4): wall-clock for 1,024 samples, same batch size, same GPU
AR is timed **with** KV cache (fair, strong baseline) and **without** (classic nanoGPT loop).

In [ ]:
def sync(): torch.cuda.synchronize() if dev.type == 'cuda' else None
def timeit(fn, n, bs, reps):
    fn(bs); sync(); ts = []                      # warmup
    for _ in range(reps):
        sync(); t = time.perf_counter()
        for i in range(0, n, bs): fn(min(bs, n - i))
        sync(); ts.append(time.perf_counter() - t)
    return min(ts)
RES['bench'] = []; nB = nS
for bs in CFG['bench_bs']:
    fns = {'NCM (K=%d NFE)' % tok.K: lambda b: ncm_sample(tok, gen, b, b, dev),
           'AR + KV cache': lambda b: ar_sample(gpt, b, b, dev, L, kv_cache=True),
           'AR no cache (nanoGPT)': lambda b: ar_sample(gpt, b, b, dev, L, kv_cache=False)}
    for name, fn in fns.items():
        t = timeit(fn, nB, bs, CFG['bench_repeats']); RES['bench'].append(dict(method=name, bs=bs, sec=t, per_sample_ms=1000*t/nB))
        print(f'bs={bs:<4} {name:<22} {t:7.2f}s for {nB} samples  ({1000*t/nB:.2f} ms/sample)')
RES['tokens_processed'] = dict(ncm=sum(tok.lens)+tok.K+L, ar_kv=L, ar_nocache=sum(range(1, L+1)))
print('token-positions processed / sample:', RES['tokens_processed'])

## 7 · Summary

In [ ]:
gh_best = [r for r in gh if r['step'] == gf['best_step']][0]
RES.update(ncm_params=gf['params'], ar_params=af['params'], tok_params=tm['params'], ncm_val_bound=gh_best['val_bound'], ar_val_nll=af['best_val_nll'],
           tok_val_recon_acc=tm['val']['acc'][-1], total_minutes=(time.time()-T_START)/60)
json.dump(RES, open(WORK/'results.json','w'), indent=1)
print(f"Runtime {RES['total_minutes']:.0f} min | NCM gen {RES['ncm_params']/1e6:.1f}M (+tokenizer {RES['tok_params']/1e6:.1f}M) | AR {RES['ar_params']/1e6:.1f}M")
print(f"Tokenizer val recon acc (full hierarchy): {RES['tok_val_recon_acc']*100:.2f}%")
print(f"Val quality (nats/token, 128-tok blocks): AR exact NLL {RES['ar_val_nll']:.3f} vs NCM ELBO upper bound {RES['ncm_val_bound']:.3f}  (ELBO is an UPPER bound on NLL, so not a like-for-like likelihood)")
b = {(r['method'], r['bs']): r['sec'] for r in RES['bench']}; nm = [m for m, _ in b if m.startswith('NCM')][0]
print('\nWall-clock for %d samples (speedup = AR time / NCM time; >1 means NCM faster):' % nB)
for bs in CFG['bench_bs']:
    print(f"  bs={bs}: vs AR+KV {b[('AR + KV cache', bs)]/b[(nm, bs)]:.2f}x | vs AR no-cache {b[('AR no cache (nanoGPT)', bs)]/b[(nm, bs)]:.2f}x")
print('\nCLAIM CHECK')
kv_ok = all(b[(nm, bs)] < b[('AR + KV cache', bs)] for bs in CFG['bench_bs'])
print(' •', 'NCM was faster than the KV-cached AR baseline at every tested batch size in THIS tiny-model/T4 setting.' if kv_ok else
      ' NCM was NOT consistently faster than the KV-cached AR baseline here -> this run does not support "NCM is faster than AR".')
print(' • "Faster than LLMs" is NOT supported: the baseline is a ~%.0fM-param AR model on 128-token blocks, not an LLM; NFE (%d vs %d) is not FLOPs or latency (NCM processes %d token-positions/sample vs %d for cached AR).' % (RES['ar_params']/1e6, tok.K, L, RES['tokens_processed']['ncm'], L))
print(' • Paper speedups are against diffusion LMs on an H100 with larger models; none of that is tested here. One seed, one dataset, 1 MB: differences are indicative only.')
print(' • Gen-PPL uses a web-text reference LM and can be gamed by low-entropy output; read it together with unigram entropy and the val bound.')